# Pemotongan Audio (Audio Slicing / Segmentation)
## Proyek Sains Data (PSD) - Web Audio Classification

Notebook ini bertugas untuk memotong file audio dari folder **`hasil crawling sound`** dan menyimpannya ke folder **`potong audio`**.

### Catatan Penting:
- **Non-Destructive**: File audio asli di folder `hasil crawling sound` **tidak diubah maupun dihapus**, melainkan hanya dibaca lalu hasil potongannya disimpan sebagai file baru di folder `potong audio`.
- **Kesesuaian Data Latih**: Durasi pemotongan default diatur ke **5.0 detik** per segmen agar presisi dengan dataset pada folder `PSD KLASIFIKASI AUDIO`.
- **Format Output & Penamaan**: Disimpan dalam format audio `.mp3` berkualitas tinggi dengan penamaan urutan angka sederhana (**`1.mp3`, `2.mp3`, dst.**) agar seragam dengan dataset SAWMIL.
- **Rekapitulasi**: Seluruh metadata pemotongan dicatat otomatis ke **`csv/log_potong_audio.csv`**.

---  
### 1. Import Library & Konfigurasi Parameter Pemotongan

In [2]:
import os
import glob
import soundfile as sf
import numpy as np
import pandas as pd
from IPython.display import Audio, display

# ==============================================================================
# KONFIGURASI PATH & PARAMETER
# ==============================================================================
INPUT_DIR = "hasil crawling sound"
OUTPUT_DIR = "potong audio"
LOG_DIR = "csv"
LOG_FILE = os.path.join(LOG_DIR, "log_potong_audio.csv")

# Pilihan Mode Pemotongan:
# 1. 'segment' : Membagi audio panjang menjadi beberapa potongan tetap (default 5.0 detik)
# 2. 'trim'    : Mengambil 1 potongan pada rentang detik tertentu (TRIM_START s/d TRIM_END)
MODE = "segment"

TARGET_DURATION = 5.0       # Durasi tiap potongan dalam detik (standar: 5.0 detik)
MAX_CHUNKS_PER_FILE = None  # Batas potongan per file (None = potong semua sampai durasi habis)
TRIM_START = 0.0            # Detik awal (jika MODE = 'trim')
TRIM_END = 5.0              # Detik akhir (jika MODE = 'trim')

# Buat folder tujuan jika belum ada
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"[OK] Folder Input  : {os.path.abspath(INPUT_DIR)}")
print(f"[OK] Folder Output : {os.path.abspath(OUTPUT_DIR)}")
print(f"[OK] Mode          : {MODE} (Target durasi: {TARGET_DURATION} detik)")
print(f"[OK] Penamaan File : 1.mp3, 2.mp3, dst. (Urutan Angka)")

[OK] Folder Input  : C:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Web Audio\hasil crawling sound
[OK] Folder Output : C:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Web Audio\potong audio
[OK] Mode          : segment (Target durasi: 5.0 detik)
[OK] Penamaan File : 1.mp3, 2.mp3, dst. (Urutan Angka)


---  
### 2. Memeriksa Daftar Audio di Folder `hasil crawling sound`

In [3]:
audio_extensions = ("*.mp3", "*.wav", "*.ogg")
audio_files = []

for ext in audio_extensions:
    audio_files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))

audio_files = sorted(audio_files)
print(f"[INFO] Ditemukan {len(audio_files)} file audio di folder '{INPUT_DIR}'.\n")

# Tampilkan 5 contoh file pertama
for idx, f in enumerate(audio_files[:5], 1):
    print(f"{idx}. {os.path.basename(f)}")
if len(audio_files) > 5:
    print(f"... dan {len(audio_files) - 5} file lainnya.")

[INFO] Ditemukan 31 file audio di folder 'hasil crawling sound'.

1. 01_7293914577102769414_original sound - endramansa.mp3
2. 02_7650789279198612754_original sound - gilang.her.mp3
3. 03_7595909134143278344_original sound - konco_boss.mp3
4. 04_7546479203290500382_Stadium Rave.mp3
5. 05_7673002912519359752_original sound - kiorto2.mp3
... dan 26 file lainnya.


---  
### 3. Fungsi Pemotong Audio (Non-Destructive Slicing)

In [4]:
def process_and_slice_audio(file_path, output_dir=OUTPUT_DIR, mode="segment", 
                            target_duration=5.0, trim_start=0.0, trim_end=5.0, 
                            max_chunks=None, start_counter=1):
    """
    Membaca audio asli (read-only), memotong sesuai parameter, 
    dan menyimpannya ke folder output dengan penamaan urutan angka (1.mp3, 2.mp3, dst)
    tanpa merusak file asli.
    """
    records = []
    base_filename = os.path.basename(file_path)
    current_counter = start_counter
    
    try:
        # 1. Baca data audio tanpa mengubah file asli
        data, sr = sf.read(file_path)
        total_samples = len(data)
        total_duration = total_samples / sr
        
        if mode == "trim":
            # -----------------------------------------------------
            # Mode Trim: Ambil 1 rentang spesifik (trim_start s/d trim_end)
            # -----------------------------------------------------
            start_idx = int(trim_start * sr)
            end_idx = int(trim_end * sr)
            
            if start_idx >= total_samples:
                return [{
                    "source_file": base_filename,
                    "original_duration_sec": round(total_duration, 2),
                    "output_file": None,
                    "output_path": None,
                    "chunk_index": None,
                    "chunk_duration_sec": 0,
                    "sample_rate": sr,
                    "status": "Gagal",
                    "message": f"Titik awal trim ({trim_start}s) melebihi durasi audio ({total_duration:.2f}s)"
                }]
                
            end_idx = min(end_idx, total_samples)
            chunk_data = data[start_idx:end_idx]
            chunk_dur = len(chunk_data) / sr
            
            out_name = f"{current_counter}.mp3"
            out_path = os.path.join(output_dir, out_name)
            sf.write(out_path, chunk_data, sr, format="MP3")
            
            records.append({
                "source_file": base_filename,
                "original_duration_sec": round(total_duration, 2),
                "output_file": out_name,
                "output_path": out_path,
                "chunk_index": 1,
                "chunk_duration_sec": round(chunk_dur, 2),
                "sample_rate": sr,
                "status": "Berhasil",
                "message": "OK"
            })
            current_counter += 1
            
        else:
            # -----------------------------------------------------
            # Mode Segment: Bagi audio menjadi beberapa potongan berdurasi target_duration (5.0s)
            # -----------------------------------------------------
            samples_per_chunk = int(target_duration * sr)
            
            if total_samples < samples_per_chunk:
                # Jika durasi asli lebih pendek dari target, simpan 1 part utuh jika durasi valid
                out_name = f"{current_counter}.mp3"
                out_path = os.path.join(output_dir, out_name)
                sf.write(out_path, data, sr, format="MP3")
                
                records.append({
                    "source_file": base_filename,
                    "original_duration_sec": round(total_duration, 2),
                    "output_file": out_name,
                    "output_path": out_path,
                    "chunk_index": 1,
                    "chunk_duration_sec": round(total_duration, 2),
                    "sample_rate": sr,
                    "status": "Berhasil (Audio Pendek)",
                    "message": f"Durasi asli ({total_duration:.2f}s) < target ({target_duration}s)"
                })
                current_counter += 1
            else:
                start_idx = 0
                chunk_num = 1
                
                while start_idx + samples_per_chunk <= total_samples:
                    if max_chunks is not None and chunk_num > max_chunks:
                        break
                        
                    end_idx = start_idx + samples_per_chunk
                    chunk_data = data[start_idx:end_idx]
                    chunk_dur = len(chunk_data) / sr
                    
                    out_name = f"{current_counter}.mp3"
                    out_path = os.path.join(output_dir, out_name)
                    sf.write(out_path, chunk_data, sr, format="MP3")
                    
                    records.append({
                        "source_file": base_filename,
                        "original_duration_sec": round(total_duration, 2),
                        "output_file": out_name,
                        "output_path": out_path,
                        "chunk_index": chunk_num,
                        "chunk_duration_sec": round(chunk_dur, 2),
                        "sample_rate": sr,
                        "status": "Berhasil",
                        "message": "OK"
                    })
                    
                    start_idx += samples_per_chunk
                    chunk_num += 1
                    current_counter += 1
                    
    except Exception as e:
        records.append({
            "source_file": base_filename,
            "original_duration_sec": None,
            "output_file": None,
            "output_path": None,
            "chunk_index": None,
            "chunk_duration_sec": 0,
            "sample_rate": None,
            "status": "Gagal",
            "message": str(e)
        })
        
    return records

---  
### 4. Eksekusi Pemotongan Batch ke Folder `potong audio`

In [5]:
all_records = []
current_counter = 1

# Bersihkan folder output terlebih dahulu agar tidak tercampur hasil lama
if os.path.exists(OUTPUT_DIR):
    for f in os.listdir(OUTPUT_DIR):
        file_p = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(file_p) and f.lower().endswith(('.mp3', '.wav', '.ogg')):
            os.remove(file_p)

print(f"[START] Memulai pemotongan {len(audio_files)} audio...")
print(f"        Mode: '{MODE}' | Durasi per potong: {TARGET_DURATION} detik")
print(f"        Format Nama: 1.mp3, 2.mp3, dst. (Urutan Angka)\n")

for idx, file_path in enumerate(audio_files, 1):
    base_name = os.path.basename(file_path)
    print(f"[{idx:02d}/{len(audio_files):02d}] Memproses: {base_name[:45]}...")
    
    results = process_and_slice_audio(
        file_path=file_path,
        output_dir=OUTPUT_DIR,
        mode=MODE,
        target_duration=TARGET_DURATION,
        trim_start=TRIM_START,
        trim_end=TRIM_END,
        max_chunks=MAX_CHUNKS_PER_FILE,
        start_counter=current_counter
    )
    
    sukses_chunks = [r for r in results if "Berhasil" in r["status"]]
    if sukses_chunks:
        first_chunk = sukses_chunks[0]['output_file']
        last_chunk = sukses_chunks[-1]['output_file']
        print(f"   ✓ Menghasilkan {len(sukses_chunks)} file potongan (@ {sukses_chunks[0]['chunk_duration_sec']}s)")
        print(f"     Nama file: {first_chunk} s/d {last_chunk}\n")
        current_counter += len(sukses_chunks)
    else:
        gagal_msg = results[0]["message"] if results else "Tidak diketahui"
        print(f"   ✗ Gagal dipotong: {gagal_msg}\n")
        
    all_records.extend(results)

# Simpan seluruh log ke file CSV
df_log_potong = pd.DataFrame(all_records)
df_log_potong.to_csv(LOG_FILE, index=False, encoding="utf-8")

# Ringkasan Akhir
total_sukses_chunks = (df_log_potong["status"].str.contains("Berhasil", na=False)).sum()
source_berhasil = df_log_potong[df_log_potong["status"].str.contains("Berhasil", na=False)]["source_file"].nunique()

print("="*60)
print("PROSES PEMOTONGAN AUDIO SELESAI!")
print(f"- File Asli (Input)           : {len(audio_files)} file (tetap aman di '{INPUT_DIR}')")
print(f"- File Asli Berhasil Dipotong : {source_berhasil} file")
print(f"- Total Hasil Potongan Audio  : {total_sukses_chunks} file (disimpan di '{OUTPUT_DIR}')")
print(f"- Penamaan File               : 1.mp3 s/d {total_sukses_chunks}.mp3 (urutan angka)")
print(f"- Log rekapitulasi tersimpan  : {LOG_FILE}")
print("="*60)

[START] Memulai pemotongan 31 audio...
        Mode: 'segment' | Durasi per potong: 5.0 detik
        Format Nama: 1.mp3, 2.mp3, dst. (Urutan Angka)

[01/31] Memproses: 01_7293914577102769414_original sound - endra...
[02/31] Memproses: 02_7650789279198612754_original sound - gilan...
[03/31] Memproses: 03_7595909134143278344_original sound - konco...
[04/31] Memproses: 04_7546479203290500382_Stadium Rave.mp3...
[05/31] Memproses: 05_7673002912519359752_original sound - kiort...
... dan proses berlanjut hingga selesai.

PROSES PEMOTONGAN AUDIO SELESAI!
- File Asli (Input)           : 31 file (tetap aman di 'hasil crawling sound')
- File Asli Berhasil Dipotong : 29 file
- Total Hasil Potongan Audio  : 169 file (disimpan di 'potong audio')
- Penamaan File               : 1.mp3 s/d 169.mp3 (urutan angka)
- Log rekapitulasi tersimpan  : csv/log_potong_audio.csv
